In [19]:
# 1-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_1.csv')
# print(df.info())
# print(df.head())

df1=df.groupby('소주제')['정답여부'].sum() / df.groupby('소주제')['정답여부'].count()
df1.sort_values(ascending=False).rank(method='dense', ascending=False)
df1['경제']

np.float64(0.68)

In [33]:
# 1-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_2.csv')
# print(df.info())
# print(df.head())
df['year']=df['date'].str.split('-').str[0]
df['month']=df['date'].str.split('-').str[1]
# print(df.info())
# print(df.head())
print(df.groupby(['year', 'month'])['price'].sum().sort_values(ascending=False)[2])
# df.groupby(['year', 'month'])['price'].sum().sort_values(ascending=False).index[3]
print(df[(df['year']=='2024') & (df['month']=='12')].groupby('category')['price'].sum().sort_values(ascending=False).max())

1738018
1012500


C:\Users\user\AppData\Local\Temp\ipykernel_3268\1317802034.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(df.groupby(['year', 'month'])['price'].sum().sort_values(ascending=False)[2])


In [48]:
# 1-3
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_3.csv')
# print(df.info())
# print(df.head())

df['a']=df['message'].str.split(' ')
# print(df.info())
# print(df.head())
df1=df[df['label']=='spam']['a']
df2=df[df['label']=='ham']['a']
df1, df2
def count(x):
    return len(x)
a=df1.apply(count).mean()
b=df2.apply(count).mean()
print(np.round(np.abs(a-b),3))

0.3


In [102]:
# 2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_test.csv')
# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

train_df['gas_totl']=train_df['gas_totl'].replace(0, np.nan)
train_df=train_df.dropna()

# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

train_y=train_df['gas_totl']
train_X=train_df.drop(columns=['gas_totl'])
test_X=test_df.drop(columns=['gas_totl'])

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

train_X, valid_X, train_y, valid_y=train_test_split(train_X, train_y, test_size=0.3)
# print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

obj=train_X.select_dtypes('object').columns.tolist()
num=train_X.select_dtypes('number').columns.tolist()
# print(obj, num)

encoder=OneHotEncoder(sparse_output=False, handle_unknown='ignore')
train_X_encoder=encoder.fit_transform(train_X[obj])
valid_X_encoder=encoder.transform(valid_X[obj])
test_X_encoder=encoder.transform(test_X[obj])

train_X=np.concatenate([train_X_encoder, train_X[num]],axis=1)
valid_X=np.concatenate([valid_X_encoder, valid_X[num]],axis=1)
test_X=np.concatenate([test_X_encoder, test_X[num]],axis=1)

rf=RandomForestRegressor()
# rf.fit(train_X, train_y)
# pred1=rf.predict(valid_X)
# print((root_mean_squared_error(valid_y, pred1) / valid_y.mean())*100)
# print(root_mean_squared_error(valid_y, pred1))

params={
    'max_depth': [10,20,30],
    'min_samples_split': [2,5,10]
}
cv=GridSearchCV(estimator=rf, param_grid=params, scoring='neg_root_mean_squared_error', cv=3)
# cv.fit(train_X, train_y)
# pred2=cv.predict(valid_X)
# print(root_mean_squared_error(valid_y, pred2))

train_X_full=np.concatenate([train_X, valid_X],axis=0)
train_y_full=np.concatenate([train_y, valid_y],axis=0)

rf.fit(train_X_full, train_y_full)

cv.fit(train_X_full, train_y_full)
pred3=cv.predict(test_X)

pd.DataFrame(pred3, columns=['pred']).to_csv('result.csv', index=False)

In [60]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_3_1.csv')
# print(df.info())
# print(df.head())

import statsmodels.api as sm
import statsmodels.formula.api as smf
model=smf.glm('attrition ~ age + income + overtime', data=df, family=sm.families.Binomial()).fit()
# print(model.summary())
model.pvalues[1:] < 0.05
print(np.round(model.params['income'],3))
print(np.round(np.exp(model.params['age']),3))
test_X=pd.DataFrame({'age': [20], 'income': [3000], 'overtime': [2]})
pred1=model.predict(test_X)
print(np.round(pred1[0],3))

-0.005
0.905
0.431


In [77]:
# 3-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_3_2.csv')
# print(df.info())
# print(df.head())
import statsmodels.api as sm
import statsmodels.formula.api as smf
model=smf.ols('price ~ area + height + wall', data=df).fit()
# print(model.summary())
model.pvalues[1:] < 0.05
print(np.round(model.params[['area', 'height']].sum(),3))
model1=smf.ols('price ~ area + height', data=df).fit()
# print(dir(model1))
print(np.round(model1.rsquared,3))
test_X=pd.DataFrame({'area': [100], 'height': [10], 'wall': [1]})
pred1=model1.predict(test_X)
print(np.round(pred1[0],3))

10.289
0.859
329.036
